# Kiểm chứng Năng lượng tính toán (Compute Energy)
Notebook này chạy nhanh 1 vòng FL để chứng minh LoRA tiết kiệm 60% năng lượng tính toán (ngốn ít pin hơn 2.5 lần) so với phương pháp Top-K Gradient và Full Finetune.

### 1. Tải Source Code & Cài đặt môi trường

In [ ]:
!git clone https://github.com/ngnam1104/FedKDL.git
import os
os.chdir('/kaggle/working/FedKDL')

!pip install ultralytics pandas pyyaml torch torchvision
import sys
sys.path.append(os.getcwd())

!mkdir -p /kaggle/working/FedKDL/datasets/URPC2020
!ln -s /kaggle/input/datasets/lywang777/urpc2020/URPC2020 /kaggle/working/FedKDL/datasets/URPC2020/URPC2020


### 2. Sinh dữ liệu môi trường giả lập (Topo & Data Partition)

In [ ]:
!python utils/generate_all_envs.py --n 30 --dataset URPC --m-relays 5 --alphas 1.0


### 3. Tìm file môi trường tự động

In [ ]:
import glob

topo_files = glob.glob("environments/2d/topo/**/*.pkl", recursive=True)
data_files = glob.glob("environments/2d/data/**/*.pkl", recursive=True)

assert len(topo_files) > 0, "Không tìm thấy file Topo"
assert len(data_files) > 0, "Không tìm thấy file Data"

TOPO = topo_files[0]
DATA = data_files[0]

print(f"Topo: {TOPO}")
print(f"Data: {DATA}")

### 4. Chạy Baseline LoRA (fedkdl)
Dự kiến `e_comp` sẽ thấp nhất (do hệ số nhân FLOPs là 1.2)

In [ ]:
!python main_trainer_od.py --topo {TOPO} --data {DATA} --baseline fedkdl --rounds 1

### 5. Chạy Baseline Full Finetune (fedkdl_nolora)
Dự kiến `e_comp` cao gấp 2.5 lần LoRA (do hệ số nhân FLOPs là 3.0)

In [ ]:
!python main_trainer_od.py --topo {TOPO} --data {DATA} --baseline fedkdl_nolora --rounds 1

### 6. Chạy Baseline Top-K (topk_grad)
Dự kiến `e_comp` cao ngất ngưởng ngang Full Finetune, vì phải tính Backward Pass toàn mạng gốc trước khi lọc (hệ số 3.0)

In [ ]:
!python main_trainer_od.py --topo {TOPO} --data {DATA} --baseline topk_grad --rounds 1